### Building Chatbot with Multiple Tools "USING" Langgraph

#### Aim
Create a chatbot with tool capabilities from arxiv, wikipedia search and some functions

In [37]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [38]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [39]:
arxiv.invoke("Attention is all you need")


HTTPError: Page request resulted in HTTP 301: None (http://export.arxiv.org/api/query?search_query=Attention+is+all+you+need&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=2)

In [40]:

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name


'wikipedia'

In [41]:
wiki.invoke("What is Machine LEARNING")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [42]:
from dotenv import load_dotenv
load_dotenv()
import os 
os.environ ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [43]:
## tavily search tools 
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(max_results=10)

In [44]:
tavily_tool.invoke("Provide me the recent ai News.")


[{'title': 'AI News | Latest News | Insights Powering AI-Driven Business ...',
  'url': 'https://www.artificialintelligence-news.com',
  'content': 'Explore More\n\n# Hershey applies AI across its supply chain operations\n\n# With higher risk and broader impact, AI security is the new supply chain problem\n\n# Gatik raises $200M to scale AI-powered autonomous freight\n\n# OneRail uses Nvidia AI for real-time last-mile delivery optimisation\n\n#### Applications\n\n### Thailand becomes one of the first in Asia to get the Sora app\n\nEntertainment & Media\n\nOctober 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration [...] Artificial Int

In [45]:
### cobine all the tools in the list 
tools=[arxiv,wiki,tavily_tool]

In [46]:
## Initialize my llm models
from langchain_groq import  ChatGroq
llm = ChatGroq(model="gpt-oss-20b") 
llm_with_tools=llm.bind_tools(tools)

In [47]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

In [48]:
llm_with_tools.invoke([HumanMessage(content="What is the recent AI news?")])


NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-oss-20b` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [58]:
## State Schema
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages
class State(TypedDict):
	messages: Annotated[list[AnyMessage], add_messages]


In [60]:
### Node definition

def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

# Build graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", END)

In [61]:
from IPython.display import display, Image

# View
display(Image(graph.get_graph().draw_mermaid_png()))

NameError: name 'graph' is not defined

In [ ]:

messages=graph.invoke({"messages": HumanMessage(content="What is attention is all you need")})
for m in messages['messages']:
    m.pretty_print()